# 05 - PostgreSQL Lakehouse Federation: Vehicle Reference Enrichment

**Project:** Auto Insurance Claims & Telematics Analytics

## What this notebook does
Uses Databricks Lakehouse Federation to join claims data against a vehicle
reference table hosted in an external PostgreSQL database (Neon, now a
Databricks-owned product) - live query federation, zero data copying.
Formalized as a gold table in the medallion pipeline.

## Setup (done via UI, not code)
- PostgreSQL database hosted on Neon (free tier)
- Small vehicle_reference table (10 make/model combinations) created directly
  in Neon's SQL editor
- Lakehouse Federation connection + foreign catalog (neon_auto_insurance_catalog)
  created via Catalog Explorer

## Table created
- `main.auto_insurance_telematics.gold_claims_enriched_with_vehicle_reference`

## Coverage note
Reference table covers 10 of 39 distinct make/model combinations in
silver_claims (25.6% of distinct combinations), which explains the 26.9%
claim-level match rate - not a coincidence, since combinations repeat evenly
across claims (~25-26 claims per combination on average). This is a
proof-of-concept reference table, not a comprehensive vehicle master data
set - a production version would cover all make/model combinations.

In [0]:
%sql
SELECT * FROM neon_auto_insurance_catalog.public.vehicle_reference;

auto_make,auto_model,vehicle_class,typical_msrp_usd
Saab,92x,Compact,25000
Mercedes,E400,Luxury Sedan,55000
Dodge,RAM,Pickup Truck,35000
Chevrolet,Tahoe,SUV,52000
Accura,RSX,Compact,24000
Toyota,Camry,Sedan,27000
Honda,CRV,SUV,30000
Ford,F150,Pickup Truck,38000
BMW,X5,Luxury SUV,62000
Audi,A3,Compact,34000


In [0]:
%sql
CREATE OR REPLACE TABLE main.auto_insurance_telematics.gold_claims_enriched_with_vehicle_reference
COMMENT 'Claims data enriched with vehicle class and MSRP via PostgreSQL Lakehouse Federation (Neon). LEFT JOIN preserves all 1,000 claims - vehicle_class and typical_msrp_usd are NULL where no reference match exists (73.1% of claims, since the reference table is a 10-combination proof-of-concept covering 25.6% of the 39 distinct make/model combinations present in the claims data, not a comprehensive vehicle catalog).'
AS
SELECT
  c.policy_number,
  c.auto_make,
  c.auto_model,
  c.auto_year,
  c.incident_severity,
  c.total_claim_amount,
  c.fraud_reported,
  r.vehicle_class,
  r.typical_msrp_usd
FROM main.auto_insurance_telematics.silver_claims c
LEFT JOIN neon_auto_insurance_catalog.public.vehicle_reference r
  ON c.auto_make = r.auto_make AND c.auto_model = r.auto_model;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(vehicle_class) AS rows_with_match
FROM main.auto_insurance_telematics.gold_claims_enriched_with_vehicle_reference;

total_rows,rows_with_match
1000,269


In [0]:
%sql
SELECT
  c.policy_number,
  c.auto_make,
  c.auto_model,
  c.total_claim_amount,
  r.vehicle_class,
  r.typical_msrp_usd
FROM main.auto_insurance_telematics.silver_claims c
LEFT JOIN neon_auto_insurance_catalog.public.vehicle_reference r
  ON c.auto_make = r.auto_make AND c.auto_model = r.auto_model
WHERE r.vehicle_class IS NOT NULL
LIMIT 10;

policy_number,auto_make,auto_model,total_claim_amount,vehicle_class,typical_msrp_usd
645723,Saab,92x,3300,Compact,25000
556080,Mercedes,E400,5060,Luxury Sedan,55000
564654,Dodge,RAM,48000,Pickup Truck,35000
836349,Chevrolet,Tahoe,60320,SUV,52000
889764,Accura,RSX,70400,Compact,24000
573572,Toyota,Camry,57200,Sedan,27000
354455,Honda,CRV,45180,SUV,30000
219028,Ford,F150,52650,Pickup Truck,38000
221186,BMW,X5,64400,Luxury SUV,62000
227244,Audi,A3,89520,Compact,34000


In [0]:
%sql
SELECT
  c.policy_number,
  c.auto_make,
  c.auto_model,
  c.total_claim_amount,
  r.vehicle_class,
  r.typical_msrp_usd
FROM main.auto_insurance_telematics.silver_claims c
LEFT JOIN neon_auto_insurance_catalog.public.vehicle_reference r
  ON c.auto_make = r.auto_make AND c.auto_model = r.auto_model
WHERE r.vehicle_class IS NOT NULL
LIMIT 10;

policy_number,auto_make,auto_model,total_claim_amount,vehicle_class,typical_msrp_usd
645723,Saab,92x,3300,Compact,25000
556080,Mercedes,E400,5060,Luxury Sedan,55000
564654,Dodge,RAM,48000,Pickup Truck,35000
836349,Chevrolet,Tahoe,60320,SUV,52000
889764,Accura,RSX,70400,Compact,24000
573572,Toyota,Camry,57200,Sedan,27000
354455,Honda,CRV,45180,SUV,30000
219028,Ford,F150,52650,Pickup Truck,38000
221186,BMW,X5,64400,Luxury SUV,62000
227244,Audi,A3,89520,Compact,34000


In [0]:
%sql
SELECT
  COUNT(*) AS total_claims,
  COUNT(r.vehicle_class) AS claims_with_reference_match,
  ROUND(COUNT(r.vehicle_class) * 100.0 / COUNT(*), 1) AS pct_matched
FROM main.auto_insurance_telematics.silver_claims c
LEFT JOIN neon_auto_insurance_catalog.public.vehicle_reference r
  ON c.auto_make = r.auto_make AND c.auto_model = r.auto_model;

total_claims,claims_with_reference_match,pct_matched
1000,269,26.9


In [0]:
%sql
SELECT COUNT(DISTINCT CONCAT(auto_make, ' ', auto_model)) AS distinct_make_models
FROM main.auto_insurance_telematics.silver_claims;

distinct_make_models
39
